In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision.datasets import CIFAR10 
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

In [2]:
transfrom = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ]
)

trainset = CIFAR10(root = "./data", train = True, download = True, transform = transfrom)
testset = CIFAR10(root = "./data", train = False, download = True, transform = transfrom)

In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
testset

Dataset CIFAR10
    Number of datapoints: 10000
    Root location: ./data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [5]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

## Build the CNN Model

In [6]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)
        )

        self.fc_layer = nn.Sequential(
            nn.Linear(4 * 4 * 128, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layer(x)
        return x

In [7]:
model = CNN()

In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Training The CNN Model

In [12]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"Epoch = {epoch+1}/{epochs} & Loss = {epoch_training_loss/len(trainloader)}")

Epoch = 1/10 & Loss = 0.13195236224700194
Epoch = 2/10 & Loss = 0.1142614868688671
Epoch = 3/10 & Loss = 0.09746759399876494
Epoch = 4/10 & Loss = 0.08698972728629799
Epoch = 5/10 & Loss = 0.08293566188139036
Epoch = 6/10 & Loss = 0.07955704718921096
Epoch = 7/10 & Loss = 0.07119078483120503
Epoch = 8/10 & Loss = 0.07154524654311978
Epoch = 9/10 & Loss = 0.06338764775140673
Epoch = 10/10 & Loss = 0.07562188619031401


In [22]:
# Evaluation

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

accuracy = 100 * correct_labels / total_labels

print(f"Accuracy = {correct_labels/total_labels*100}")

Accuracy = 74.92999999999999
